In [1]:
# Import required libraries
import os
import pickle
import sys
import numpy as np
from sklearn.model_selection import KFold
import torch
from tabpfn import TabPFNRegressor
from sklearn.metrics import mean_squared_error as mse
import time
# Add the parent directory to the path to import helper modules
sys.path.append("../")
from helper import load_data, preprocess, data_source_release
from src.distnet_torch import DistNetModel, nllh_loss_torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("All imports successful!")

Using device: cuda
All imports successful!


In [2]:
# Configuration parameters (equivalent to command line arguments)
class Args:
    def __init__(self):
        # Get available scenarios
        sc_dict = data_source_release.get_sc_dict()
        print("Available scenarios:", list(sc_dict.keys()))
        
        # Set default parameters
        self.scenario = 'clasp_factoring'  # Use first available scenario
        self.num_train_samples = 100
        self.fold = 0  # Must be between 0-9
        self.save = "../tabpfn/results"  # Directory to save results
        self.seed = 100
        
        print(f"Configuration:")
        print(f"  Scenario: {self.scenario}")
        print(f"  Fold: {self.fold}")
        print(f"  Training samples: {self.num_train_samples}")

args = Args()

Available scenarios: ['clasp_factoring', 'saps-CVVAR', 'spear_qcp', 'yalsat_qcp', 'spear_swgcp', 'yalsat_swgcp', 'lpg-zeno']
Configuration:
  Scenario: clasp_factoring
  Fold: 0
  Training samples: 100


In [3]:
# Helper function to save results
def dump_res(save_path, tra_pred, val_pred, add_info):
    """Save the training and validation predictions along with additional info"""
    with open(save_path, "wb") as fh:
        pickle.dump([tra_pred, val_pred, add_info], fh,
                    protocol=pickle.HIGHEST_PROTOCOL)
    print("Dumped to %s" % save_path)

In [4]:
# Setup paths and info - fixed for lognormal_nn.floc
# Assertions
assert 0 <= args.fold <= 9

model_name = "tabpfn_v2"

# Create directory if it does not exist
if not os.path.exists(args.save):
    os.makedirs(args.save)

save_path = os.path.join(args.save, "%s.%s.%d.%d.pkl" % (args.scenario,
                                                            model_name,
                                                            args.fold,
                                                            args.seed))
if args.num_train_samples != 100:
    save_path += "_%d" % args.num_train_samples

add_info = {"scenario": args.scenario,
            "fold": args.fold, "model": model_name, "loaded": False,
            "num_train_samples": args.num_train_samples,
            "seed": args.seed}

print("Save path:", save_path)
print("Additional info:", add_info)

Save path: ../tabpfn/results/clasp_factoring.tabpfn_v2.0.100.pkl
Additional info: {'scenario': 'clasp_factoring', 'fold': 0, 'model': 'tabpfn_v2', 'loaded': False, 'num_train_samples': 100, 'seed': 100}


In [5]:
# Load data
sc_dict = data_source_release.get_sc_dict()
data_dir = data_source_release.get_data_dir()

print("Loading data for scenario:", args.scenario)
print("Data directory:", data_dir)

runtimes, features, sat_ls = load_data.get_data(
    scenario=args.scenario, 
    data_dir=data_dir,
    sc_dict=sc_dict, 
    retrieve=sc_dict[args.scenario]['use']
)

features = np.array(features)
runtimes = np.array(runtimes)

print("Data shapes:")
print("Runtimes shape:", runtimes.shape)
print("Features shape:", features.shape)

Loading data for scenario: clasp_factoring
Data directory: /home/ihagverdi/DistNet-DL-LAB-Freiburg/notebooks/../data
Train data loaded
Test data loaded
/home/ihagverdi/DistNet-DL-LAB-Freiburg/notebooks/../data/clasp-3.0.4-p8_rand_factoring/features.txt
(2000, 113) (2000, 100)
Discarding 0 (2000) instances because of CRASHED
Discarding 0 (2000) instances because of TIMEOUT
Discarding 0 (2000) instances because not stated TIMEOUTS
Discarding 0 (2000) instances because of constant features
Discarding 0 (2000) instances because of UNSAT
(2000, 113)
Data shapes:
Runtimes shape: (2000, 100)
Features shape: (2000, 113)


In [6]:
# Extract probability distributions from TabPFN predictions
def get_nllh(model, predictions, true_values, device):
    """
    Extract probability density values from TabPFN predictions for validation data.
    
    Args:
        model: Trained TabPFN model
        predictions: Full predictions from model.predict()
        true_values: True validation values
        device: PyTorch device (CPU/GPU)
    
    Returns:
        dict: Dictionary containing probability calculations
    """

    # get criterion
    criterion = predictions['criterion']

    # Extract logits from predictions
    prediction_logits = torch.as_tensor(predictions['logits'], dtype=torch.float32).to(device)

    # Convert validation targets to tensor on appropriate device
    validation_targets_tensor = torch.as_tensor(true_values, dtype=torch.float32).to(device)

    # Calculate probability density function values
    pdf_values = criterion.pdf(prediction_logits, validation_targets_tensor)

    nllh = -torch.log(pdf_values)

    return nllh.detach().cpu().numpy()  # Return negative log likelihood

### Given a dataset, run tabpfn on 10 folds and return the average negative log likelihood.

In [7]:
args.num_train_samples = 100
N_SUBSAMPLE_TRAIN = 4096
N_SUBSAMPLE_VAL = 4096
DO_SUBSAMPLE = True  # Set to True to enable subsampling

In [ ]:
avg_nllh = []
idx = list(range(runtimes.shape[0]))
kf = KFold(n_splits=10, shuffle=True, random_state=0)
for fold in range(10):
    cntr = -1
    for train, valid in kf.split(idx):
        # Reset seed for every instance
        np.random.seed(2)
        cntr += 1
        if cntr != fold:
            continue

        X_train = features[train, :]
        X_valid = features[valid, :]

        y_train = runtimes[train]
        y_valid = runtimes[valid]

        X_train, X_valid = preprocess.preprocess_features(X_train, X_valid,
                                                            scal="meanstd")

        print("Evaluating %s, %s, %s on %s" % (args.scenario, model_name,
                                                    str(X_train.shape),
                                                    str(y_train.shape)))

        # Prepare flattened data
        X_trn_flat = np.concatenate(
            [[x for i in range(100)] for x in X_train])
        X_vld_flat = np.concatenate(
            [[x for i in range(100)] for x in X_valid])
        y_trn_flat = y_train.flatten().reshape([-1, 1])
        y_vld_flat = y_valid.flatten().reshape([-1, 1])

        # Unfold data
        subset_idx = list(range(100))
        if args.num_train_samples != 100:
            print("Cut data down to %d samples with seed %d" %
                    (args.num_train_samples, args.seed))
            rs = np.random.RandomState(args.seed)
            rs.shuffle(subset_idx)
            subset_idx = subset_idx[:args.num_train_samples]

            # Only shorten data used for training
            X_trn_flat = np.concatenate(
                [[x for i in range(args.num_train_samples)] for x in X_train])
            y_train = y_train[:, subset_idx]
            y_trn_flat = y_train.flatten().reshape([-1, 1])

            X_vld_flat = np.concatenate(
                [[x for i in range(args.num_train_samples)] for x in X_valid])
            y_valid = y_valid[:, subset_idx]
            y_vld_flat = y_valid.flatten().reshape([-1, 1])

        # Min/Max Scale runtimes
        y_max_ = np.max(y_trn_flat)
        y_min_ = 0

        y_trn_flat = (y_trn_flat - y_min_) / y_max_
        y_vld_flat = (y_vld_flat - y_min_) / y_max_

        y_train = (y_train - y_min_) / y_max_
        y_valid = (y_valid - y_min_) / y_max_

        # rename for ease of use
        X_train = X_trn_flat
        y_train = y_trn_flat.ravel()

        X_val = X_vld_flat
        y_val = y_vld_flat.ravel()

        print(f"X_train shape: {X_train.shape}")
        print(f"y_train shape: {y_train.shape}")
        print(f"X_val shape: {X_val.shape}")
        print(f"y_val shape: {y_val.shape}")

        if DO_SUBSAMPLE:
            rs = np.random.RandomState(args.seed)
            subsample_train_idx = rs.choice(len(X_train), size=N_SUBSAMPLE_TRAIN, replace=True)  # iid random sampling

            X_train = X_train[subsample_train_idx]
            y_train = y_train[subsample_train_idx]

            print(f"I.I.D Subsampling to {N_SUBSAMPLE_TRAIN} samples. New X_train shape: {X_train.shape}")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Using device:", device)
        model = TabPFNRegressor(device=device, ignore_pretraining_limits=True)
        start = time.time()
        model.fit(X_train, y_train)

        # batched validation nllh
        val_batch_size = N_SUBSAMPLE_VAL  # Use the defined validation batch size
        n_val_samples = X_val.shape[0]
        
        all_nllh_values = []
        
        print(f"Processing validation set in batches of {val_batch_size}")
        with torch.no_grad():
            for batch_start in range(0, n_val_samples, val_batch_size):
                batch_end = min(batch_start + val_batch_size, n_val_samples)
                
                # Get current batch
                X_val_batch = X_val[batch_start:batch_end]
                y_val_batch = y_val[batch_start:batch_end]
                
                print(f"Processing batch {batch_start//val_batch_size + 1}: samples {batch_start} to {batch_end-1}")
                
                # Get predictions for this batch
                full_pred_batch = model.predict(X_val_batch, output_type='full')
                
                # Calculate NLL for this batch
                nllh_batch = get_nllh(model, full_pred_batch, y_val_batch, device)
                
                # Store the NLL values for this batch
                all_nllh_values.extend(nllh_batch)

                # clean up
                del full_pred_batch, nllh_batch
                torch.cuda.empty_cache()

        time_taken = time.time() - start
        # Convert to numpy array and calculate mean NLL
        all_nllh_values = np.array(all_nllh_values)
        nllh = np.mean(all_nllh_values)
        avg_nllh.append(nllh)
        print(f"Finished fitting tabpfn_v2 on data with {X_train.shape[0]} rows, it took {time_taken} seconds.")
        print(f"Processed {len(all_nllh_values)} validation samples in {(n_val_samples + val_batch_size - 1) // val_batch_size} batches")
        print(f"NLLH for fold {fold}: {nllh}")

avg_nllh = np.mean(avg_nllh)
print(f"Average NLLH: {avg_nllh}")

Discarding 11 (113) features
Evaluating clasp_factoring, tabpfn_v2, (1800, 102) on (1800, 100)
X_train shape: (180000, 102)
y_train shape: (180000,)
X_val shape: (20000, 102)
y_val shape: (20000,)
I.I.D Subsampling to 4096 samples. New X_train shape: (4096, 102)
Using device: cuda
Training NLLH: 1.0423128604888916
Processing validation set in batches of 4096
Processing batch 1: samples 0 to 4095


In [ ]:
# Save results
print("Saving results...")

dump_res(tra_pred=tra_pred, val_pred=val_pred,
         save_path=save_path, add_info=add_info)

print("Evaluation complete!")
print(f"Results saved to: {save_path}")

# Display some basic statistics
print("\n=== Results Summary ===")
print(f"Training predictions - Min: {tra_pred.min():.6f}, Max: {tra_pred.max():.6f}, Mean: {tra_pred.mean():.6f}")
print(f"Validation predictions - Min: {val_pred.min():.6f}, Max: {val_pred.max():.6f}, Mean: {val_pred.mean():.6f}")

# Check if predictions are finite
print(f"Training predictions finite: {np.isfinite(tra_pred).all()}")
print(f"Validation predictions finite: {np.isfinite(val_pred).all()}")